# VDB1 빌드 — 텍스트 청크 벡터DB

SQLite `text_blocks` → Qwen3-Embedding-8B (ollama) → ChromaDB

In [1]:
!pip install chromadb -q


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [12]:
import sqlite3
import ollama
import chromadb
from tqdm import tqdm

In [13]:
SQLITE_PATH = "./audit_text.db"       # 실제 DB 경로로 수정
CHROMA_PATH = "./chroma_db"
COLLECTION  = "vdb1_text"
EMBED_MODEL = "qwen3-embedding:8b"
BATCH_SIZE  = 8

## 1. SQLite에서 text_blocks 로드

In [14]:
conn = sqlite3.connect(SQLITE_PATH)
cur  = conn.cursor()

cur.execute("""
    SELECT
        tb.block_pk,
        tb.doc_id,
        d.year,
        tb.section_id,
        tb.note_id,
        tb.note_no,
        tb.note_title,
        tb.subsection_path,
        tb.block_type,
        tb.block_order,
        tb.text
    FROM text_blocks tb
    JOIN documents d ON tb.doc_id = d.doc_id
    WHERE tb.text IS NOT NULL
      AND LENGTH(TRIM(tb.text)) >
    ORDER BY d.year, tb.block_order
""")

rows = cur.fetchall()
conn.close()
print(f"총 {len(rows)}개 블록 로드")

총 2003개 블록 로드


## 2. ChromaDB 컬렉션 생성

In [15]:
client = chromadb.PersistentClient(path=CHROMA_PATH)

try:
    client.delete_collection(COLLECTION)
    print("기존 컬렉션 삭제")
except:
    pass

collection = client.create_collection(
    name=COLLECTION,
    metadata={"hnsw:space": "cosine"},
)
print(f"컬렉션 '{COLLECTION}' 생성")

기존 컬렉션 삭제
컬렉션 'vdb1_text' 생성


## 3. 임베딩 & 삽입

In [16]:
for i in tqdm(range(0, len(rows), BATCH_SIZE), desc="임베딩 중"):
    batch = rows[i : i + BATCH_SIZE]

    ids       = [str(r[0]) for r in batch]
    documents = [r[10] for r in batch]
    metadatas = [
        {
            "doc_id":          r[1],
            "year":            r[2],
            "section_id":      r[3] or "",
            "note_id":         r[4] or "",
            "note_no":         r[5] or "",
            "note_title":      r[6] or "",
            "subsection_path": r[7] or "",
            "block_type":      r[8] or "",
            "block_order":     r[9],
        }
        for r in batch
    ]

    resp       = ollama.embed(model=EMBED_MODEL, input=documents)
    embeddings = resp.embeddings

    collection.add(
        ids=ids,
        documents=documents,
        embeddings=embeddings,
        metadatas=metadatas,
    )

print(f"\n완료 — 총 {collection.count()}개 벡터 저장")

## 4. 검색 테스트

In [27]:
def search(query: str, year: int, k: int = 5):
    query_emb = ollama.embed(model=EMBED_MODEL, input=[query]).embeddings

    results = collection.query(
        query_embeddings=query_emb,
        n_results=k,
        where={"year": year},
    )

    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        print(f"[{meta['year']}년 / {meta['note_title'] or meta['block_type']}] (score: {1-dist:.3f})")
        print(f"  {doc[:12000]}...\n")


search("만약 자산 장부금액 > 추정 회수가능액 이면 어떻게돼?", year=2014)

[2014년 / 중요한 회계처리방침, 계속 :] (score: 0.573)
  2.10 매각예정분류자산집단
비유동자산(또는 처분자산집단)은 장부금액이 매각거래를 통하여 주로 회수되고, 매각될 가능성이 매우 높은 경우에 매각예정으로 분류되며, 그러한 자산은 장부금액과 순공정가치 중 작은 금액으로 측정됩니다. 계속;...

[2014년 / 특수관계자와의 거래, 계속 : 다. 채권ㆍ채무잔액 당기말 및 전기말 현재 특수관계자에 대한 채권ㆍ채무잔액은 다음과 같습니다. (1) 당기] (score: 0.555)
  33. 특수관계자와의 거래, 계속 : 다. 채권ㆍ채무잔액 당기말 및 전기말 현재 특수관계자에 대한 채권ㆍ채무잔액은 다음과 같습니다. (1) 당기
계속;...

[2014년 / 매각예정분류(처분자산집단), 계속: 나. 매각예정으로 분류한 자산 및 부채의 내역] (score: 0.537)
  34. 매각예정분류(처분자산집단), 계속: 나. 매각예정으로 분류한 자산 및 부채의 내역
매각예정분류로 인하여 인식한 손상차손 금액은 31,219백만원입니다. 다. 매각예정으로 분류한 기타자본항목의 내역...

[2014년 / 중요한 회계처리방침, 계속 : 2.13 비금융자산의 손상] (score: 0.536)
  2. 중요한 회계처리방침, 계속 : 2.13 비금융자산의 손상
영업권이나 비한정내용연수를 가진 무형자산은 상각하지 않고 매년 손상검사를 실시하고 있습니다. 상각하는 자산의 경우 매 보고기간말에 장부금액이 회수가능하지 않을 수도 있음을 나타내는 환경의 변화나 사건이 있을 때마다 손상검사를 수행하고 있습니다. 손상차손은 회수가능액을 초과하는 장부금액만큼 인식하고 있습니다. 회수가능액은 순공정가치와 사용가치 중 큰 금액으로 결정하고 있습니다. 손상을 측정하기 위한 목적으로 자산은 별도로 식별 가능한 현금흐름을 창출하는 가장 하위 수준의집단(현금창출단위)으로 그룹화하고 있습니다. 손상차손을 인식한 경우, 영업권 이외의 비금융자산은 매 보고기간말에 손상차손의 환입가능성을 검토

In [1]:
import ollama

response = ollama.chat(
    model="qwen3:4b",  # 설치된 모델명으로 변경
    messages=[
        {"role": "user", "content": "안녕 나와 대화해보자"}
    ]
)

print(response.message.content)

ResponseError: model 'qwen3:4b' not found (status code: 404)